## Load Data

In [2]:
import numpy as np, pandas as pd, sklearn, os
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import umap
import hdbscan

from google.colab import drive
drive.mount('/content/drive')

numpy: 2.0.2 pandas: 2.2.2 sklearn: 1.6.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# load in embeddings and nutrition data
EMBED_FILENAME = "embeddings/embeddings.npy"
CSV_FILENAME = "only_english.csv"
ROOT = "/content/drive/MyDrive/"

embeddings = np.load(ROOT + EMBED_FILENAME)
df = pd.read_csv(ROOT + CSV_FILENAME)

## Preprocess nutrition data

In [7]:
numeric_cols = ['ingredient_count', 'additives_n', 'additive_density', 'monounsaturated-fat_100g', 'sugars_100g',
       'polyunsaturated-fat_100g', 'carbohydrates_100g',
       'fiber_100g', 'saturated-fat_100g']

df_num = df[numeric_cols]
X_num = df_num.to_numpy()

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

X = np.hstack([embeddings, X_num_scaled])  # feature matrix
print("feature shape:", X.shape)

Final feature shape: (272809, 393)


## Handle NaNs before PCA


In [8]:
percent_missing = df_num.isnull().sum() * 100 / len(df_num)
missing_value_df = pd.DataFrame({'column_name': df_num.columns,
                                 'percent_missing': percent_missing})
missing_value_df.sort_values('percent_missing', inplace=True)
missing_value_df

,column_name,percent_missing
ingredient_count,ingredient_count,0.0
additives_n,additives_n,0.0
additive_density,additive_density,0.0
monounsaturated-fat_100g,monounsaturated-fat_100g,0.0
sugars_100g,sugars_100g,0.0
polyunsaturated-fat_100g,polyunsaturated-fat_100g,0.0
carbohydrates_100g,carbohydrates_100g,0.0
fiber_100g,fiber_100g,0.0
saturated-fat_100g,saturated-fat_100g,0.0


## PCA

In [9]:
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)
print("PCA shape:", X_pca.shape)

PCA shape: (272809, 50)


## UMAP

In [10]:
umap_cluster = umap.UMAP(
    n_neighbors=50,
    min_dist=0.1,
    n_components=10,
    metric="cosine",
    random_state=42
)

X_umap = umap_cluster.fit_transform(X_pca)
print("UMAP (10D) shape:", X_umap.shape)



/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP (10D) shape: (272809, 10)


## HDBSCAN

In [19]:
df_labels = pd.DataFrame()

In [21]:
min_cluster_sizes = [50, 100, 150, 200]
for size in min_cluster_sizes:
  clusterer = hdbscan.HDBSCAN(
      min_cluster_size=size,        # adjust based on dataset
      min_samples=10,
      metric="euclidean"
  )

  col = f"labels_{size}"
  df_labels[col] = clusterer.fit_predict(X_umap)
  print(f"Saved {col}")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Saved labels_50


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Saved labels_100


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Saved labels_150


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Saved labels_200


In [22]:
# df_labels.to_csv("label_runs.csv")

In [28]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
df_results = pd.DataFrame()

def eval_scores(X, labels, cap=4000, random_state=42):
    rng = np.random.default_rng(random_state)
    idx = np.arange(X.shape[0])
    if len(idx) > cap:
        idx = rng.choice(idx, cap, replace=False)
    return (
        float(silhouette_score(X[idx], labels[idx])),
        float(calinski_harabasz_score(X[idx], labels[idx])),
        float(davies_bouldin_score(X[idx], labels[idx]))
    )

for col in df_labels.columns:
  sil, ch, db = eval_scores(X_umap, df_labels[col])
  res = pd.Series([sil, ch, db])
  df_results[col] = res

df_results.insert(0, "metric", ["silhouette_score", "calinski_harabasz_score", "davies_bouldin_score"])

In [29]:
display(df_results)

,metric,labels_50,labels_100,labels_150,labels_200
0,silhouette_score,-0.252686,-0.204292,-0.202148,-0.201079
1,calinski_harabasz_score,8.397349,21.947039,26.553982,33.849110
2,davies_bouldin_score,1.005194,1.044552,1.115405,1.160359
